# Streaming Responses

<img src="./images/streaming-response.png" width="800" height="600" alt="Alt text">

### Notebook Setup 

In [1]:
import os
import sys
import anthropic
import json

from dotenv import load_dotenv
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
_API_KEY = os.getenv("ANTHROPIC_API_KEY", "").strip()

# Check for the availablility of key
if (not _API_KEY) or (not isinstance(_API_KEY, str)):
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif (not _API_KEY.startswith("sk-ant-")):
    print("An API key was found, but it doesn't start sk-ant-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [3]:
client = anthropic.Anthropic(api_key = _API_KEY)
CLAUDE_HAIKU_45_MODEL="claude-haiku-4-5"

### Basic Streaming

In [4]:
# stream() returns a context manager — use it with 'with'
with client.messages.stream(
    model = CLAUDE_HAIKU_45_MODEL,
    max_tokens = 32,
    messages = [{"role": "user", "content": "Count from 1 to 10, one number per line."}]
) as stream:
    # text_stream yields one string chunk at a time as they arrive
    for text_chunk in stream.text_stream:
        print(text_chunk, end = "", flush = True)  # (flush = True) prints immediately

1
2
3
4
5
6
7
8
9
10

In [5]:
print()  # newline after stream ends
# After the stream completes, get the full response object
final = stream.get_final_message()
print(f"\nstop_reason:    {final.stop_reason}")
print(f"input_tokens:   {final.usage.input_tokens}")
print(f"output_tokens:  {final.usage.output_tokens}")



stop_reason:    end_turn
input_tokens:   21
output_tokens:  23


Two things to note.  
`flush=True` is important — without it Python buffers the output and you won't see tokens arrive in real time.  
And `stream.get_final_message()` gives you the complete Message object after the stream ends, so you can still inspect `stop_reason` and usage exactly like a non-streaming call.  

### Handling raw events

`text_stream` is a convenient shortcut, but sometimes you need to react to specific events — like knowing exactly when a new content block starts.  
This is useful when you need to build a UI that shows Claude is `"thinking"` before the first token, or when you want to render different content blocks differently (e.g. tool calls vs text).  

#### Every event type, one by one

In [6]:
with client.messages.stream(
    model = CLAUDE_HAIKU_45_MODEL,
    max_tokens = 64,
    messages = [{"role": "user", "content": "Say hello in exactly 5 words."}]
) as stream:
    for event in stream:
        name = type(event).__name__

        # ── 1. RawMessageStartEvent ──────────────────────────────────────────
        # Fires once at the very start.
        # Contains the message id, model, and initial token usage.
        # output_tokens here is always 1 (the first token was needed to
        # start the stream — it gets corrected in RawMessageDeltaEvent).
        if name == "RawMessageStartEvent":
            m = event.message
            print(f"[MessageStart]")
            print(f"  id            = {m.id}")
            print(f"  model         = {m.model}")
            print(f"  input_tokens  = {m.usage.input_tokens}")
            print(f"  output_tokens = {m.usage.output_tokens}  ← always 1 here")

        # ── 2. RawContentBlockStartEvent ────────────────────────────────────
        # Fires once per content block (text block or tool_use block).
        # A simple reply has one block (index=0).
        # If Claude calls a tool it opens a new block for the tool call.
        elif name == "RawContentBlockStartEvent":
            b = event.content_block
            print(f"\n[ContentBlockStart]")
            print(f"  index         = {event.index}")
            print(f"  block.type    = {b.type}")
            if b.type == "text":
                print(f"  block.text    = '{b.text}'  ← always empty string here")
            elif b.type == "tool_use":
                print(f"  block.name    = {b.name}")
                print(f"  block.id      = {b.id}")

        # ── 3. RawContentBlockDeltaEvent ────────────────────────────────────
        # Fires many times — once per token (text) or JSON fragment (tool).
        # delta.type is either "text_delta" or "input_json_delta".
        elif name == "RawContentBlockDeltaEvent":
            d = event.delta
            if d.type == "text_delta":
                # Each chunk is a small string — often a single word or part of one
                print(f"[Delta] text_delta       = '{d.text}'")
            elif d.type == "input_json_delta":
                # Tool arguments arrive as JSON fragments
                print(f"[Delta] input_json_delta = '{d.partial_json}'")

        # ── 4. RawContentBlockStopEvent ─────────────────────────────────────
        # Fires once when a content block is fully done.
        # index matches the block that just finished.
        elif name == "RawContentBlockStopEvent":
            print(f"\n[ContentBlockStop]")
            print(f"  index         = {event.index}")

        # ── 5. RawMessageDeltaEvent ──────────────────────────────────────────
        # Fires once near the end, just before message_stop.
        # Contains the definitive stop_reason and FINAL output token count.
        elif name == "RawMessageDeltaEvent":
            print(f"\n[MessageDelta]")
            print(f"  stop_reason   = {event.delta.stop_reason}")
            print(f"  output_tokens = {event.usage.output_tokens}  ← final count")

        # ── 6. RawMessageStopEvent ───────────────────────────────────────────
        # The very last event. No data — just signals the stream is closed.
        elif name == "RawMessageStopEvent":
            print(f"\n[MessageStop]  ← stream is fully closed")

[MessageStart]
  id            = msg_01Mox1MJ6zNdcdJbjXPVkqMQ
  model         = claude-haiku-4-5-20251001
  input_tokens  = 16
  output_tokens = 2  ← always 1 here

[ContentBlockStart]
  index         = 0
  block.type    = text
  block.text    = ''  ← always empty string here
[Delta] text_delta       = 'Hello,'
[Delta] text_delta       = ' how'
[Delta] text_delta       = ' are'
[Delta] text_delta       = ' you today'
[Delta] text_delta       = '?'

[MessageDelta]
  stop_reason   = end_turn
  output_tokens = 10  ← final count


#### How blocks and deltas relate

What actually travels over the wire?  
When you use `stream()`, the SDK is receiving a stream of Server-Sent Events (SSE) — a raw HTTP connection where the server pushes lines of text as they're ready.  

In [7]:
# Asking something that produces a thinking text block + a tool_use block
tools = [{
    "name": "get_weather",
    "description": "Get the weather for a city.",
    "input_schema": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"]
    }
}]

# blocks[index] = {"type": ..., "content": ""}
# We'll build each block's content by appending deltas to it
blocks = {}

with client.messages.stream(
    model = CLAUDE_HAIKU_45_MODEL,
    max_tokens = 1024,
    tools = tools,
    system="Always write a short sentence explaining what you are about to do before calling any tool.",
    messages=[{"role": "user", "content": "What's the weather in Rome?"}]
) as stream:
    for event in stream:
        name = type(event).__name__
        if name == "RawContentBlockStartEvent":
            b = event.content_block
            # Register a new block slot
            blocks[event.index] = {
                "type": b.type,
                "content": "",
                # for tool_use blocks, also store name and id
                "tool_name": getattr(b, "name", None),
                "tool_id":   getattr(b, "id",   None),
            }
            print(f"\n→ Block {event.index} opened  (type={b.type})")
        elif name == "RawContentBlockDeltaEvent":
            d = event.delta
            idx = event.index
            if d.type == "text_delta":
                blocks[idx]["content"] += d.text
            elif d.type == "input_json_delta":
                # Tool JSON arrives fragmented — accumulate it
                blocks[idx]["content"] += d.partial_json
        elif name == "RawContentBlockStopEvent":
            idx = event.index
            b = blocks[idx]
            print(f"→ Block {idx} closed   (type={b['type']})")
            print(f"  content = '{b['content']}'")

final = stream.get_final_message()
print(f"stop_reason: {final.stop_reason}, blocks captured: {len(blocks)}")

# ── After stream: show a clean summary of all blocks ──────────────────────
print("\n── Final blocks ──")
for idx, b in blocks.items():
    if b["type"] == "text":
        print(f"  [{idx}] text     : {b['content']}")
    elif b["type"] == "tool_use":
        print(f"  [{idx}] tool_use : {b['tool_name']}({b['content']})")
        print(f"          tool_id  : {b['tool_id']}")


→ Block 0 opened  (type=text)

→ Block 1 opened  (type=tool_use)
stop_reason: tool_use, blocks captured: 2

── Final blocks ──
  [0] text     : I'll check the weather in Rome for you.
  [1] tool_use : get_weather({"city": "Rome"})
          tool_id  : toolu_01Dyz4odzcEgD76EsXeqmhym


#### Building a reusable stream handler class

In real projects you don't want if `name == "RawContentBlockStartEvent"` scattered everywhere. Here's how to wrap it cleanly:

In [8]:
from dataclasses import dataclass, field

@dataclass
class StreamResult:
    """Clean result object produced after a stream completes."""
    text_blocks: list[str]          = field(default_factory=list)
    tool_calls:  list[dict]         = field(default_factory=list)
    stop_reason: str                = ""
    input_tokens: int               = 0
    output_tokens: int              = 0

    @property
    def text(self) -> str:
        """All text blocks joined."""
        return " ".join(self.text_blocks)

    @property
    def has_tool_calls(self) -> bool:
        return len(self.tool_calls) > 0


def handle_stream(stream) -> StreamResult:
    """
    Iterate a raw event stream and return a clean StreamResult.
    Handles both text blocks and tool_use blocks.
    """
    result = StreamResult()
    blocks = {}  # index → {"type", "content", ...}

    for event in stream:
        name = type(event).__name__

        if name == "RawMessageStartEvent":
            result.input_tokens = event.message.usage.input_tokens

        elif name == "RawContentBlockStartEvent":
            b = event.content_block
            blocks[event.index] = {
                "type":      b.type,
                "content":   "",
                "tool_name": getattr(b, "name", None),
                "tool_id":   getattr(b, "id",   None),
            }

        elif name == "RawContentBlockDeltaEvent":
            d = event.delta
            if d.type == "text_delta":
                blocks[event.index]["content"] += d.text
                # Live print as tokens arrive
                print(d.text, end="", flush=True)
            elif d.type == "input_json_delta":
                blocks[event.index]["content"] += d.partial_json

        elif name == "RawContentBlockStopEvent":
            b = blocks[event.index]
            if b["type"] == "text":
                result.text_blocks.append(b["content"])
            elif b["type"] == "tool_use":
                result.tool_calls.append({
                    "id":    b["tool_id"],
                    "name":  b["tool_name"],
                    "input": json.loads(b["content"]) if b["content"] else {}
                })

        elif name == "RawMessageDeltaEvent":
            result.stop_reason   = event.delta.stop_reason
            result.output_tokens = event.usage.output_tokens

    return result


# ── Try it out ────────────────────────────────────────────────────────────

tools = [{
    "name": "get_weather",
    "description": "Get the weather for a city.",
    "input_schema": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"]
    }
}]

print("Response: ", end="")
with client.messages.stream(
    model = CLAUDE_HAIKU_45_MODEL,
    max_tokens = 256,
    tools = tools,
    messages = [{"role": "user", "content": "What's the weather in Madrid?"}]
) as stream:
    result = handle_stream(stream)

print(f"\n\n── StreamResult ──")
print(f"text          : {result.text!r}")
print(f"has_tool_calls: {result.has_tool_calls}")
print(f"tool_calls    : {result.tool_calls}")
print(f"stop_reason   : {result.stop_reason}")
print(f"input_tokens  : {result.input_tokens}")
print(f"output_tokens : {result.output_tokens}")

Response: 

── StreamResult ──
text          : ''
has_tool_calls: False
tool_calls    : []
stop_reason   : tool_use
input_tokens  : 564
output_tokens : 54


|           Event           |      Fires     |                   What you get                  |
|:-------------------------:|:--------------:|:-----------------------------------------------:|
| RawMessageStartEvent      | Once           | id, model, initial input_tokens                 |
| RawContentBlockStartEvent | Once per block | index, type (text or tool_use), tool name/id    |
| RawContentBlockDeltaEvent | Many times     | text_delta chunks or input_json_delta fragments |
| RawContentBlockStopEvent  | Once per block | index (block is now complete)                   |
| RawMessageDeltaEvent      | Once           | stop_reason, final output_tokens                |
| RawMessageStopEvent       | Once           | nothing — just signals stream is closed         |